# 08 · E2: train the answerability adapter (Step 3.3)

**Runtime: L4 GPU** (Runtime → Change runtime type → **L4 GPU**).

It learns to reply ANSWERABLE or UNANSWERABLE: can the question be answered from this photo?

1. **Train**: bf16 LoRA, same settings as the answering adapter (D-051), 17,967 training questions (all of them, answerable and not), 1 epoch (1123 optimizer steps), 524,288 px. Checkpoint every 10 minutes to Drive; live curve in W&B.
2. **Evaluate** on the full test set (4,319 test questions), same pipeline as E1.
3. **Compare** with E1 on the same examples (paired bootstrap).

Expect **about 4 hours**, almost all of it training. Keep the tab open.

**If Colab disconnects:** reconnect (L4) and **Run all** again. Training resumes from the last checkpoint (at most 10 minutes lost); finished parts are skipped.

Every command runs through `sh()`, which stops with an error if the command fails.

## 1 · Settings, Drive, code, W&B key

In [ ]:
import json
import os
import subprocess

from google.colab import drive, userdata

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
BRANCH = "main"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
DATA = f"{DRIVE_ROOT}/datasets"
LOCAL = "/content/data"
CKPT = f"{DRIVE_ROOT}/checkpoints"
RUNS = f"{DRIVE_ROOT}/runs"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"
TASK = "answerability"
RUN_ID = "e2-answerability-lora"
ADAPTER = f"{CKPT}/{RUN_ID}/adapter"


def sh(cmd, check=True, **env):
    """Run a shell command, stream its output, and raise if it fails (unless check=False)."""
    proc = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, **env},
    )
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if check and code:
        raise RuntimeError(f"Command failed (exit {code}). Stop here and paste the output above.")
    return code


try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")  # never printed
    print("W&B key loaded from Colab Secrets.")
except Exception as e:
    raise RuntimeError(
        "Add WANDB_API_KEY in the key icon (Secrets) on the left, turn on notebook access, then rerun."
    ) from e
os.environ["WANDB_PROJECT"] = "vizwiz-assistant"
os.environ["HF_HOME"] = "/content/hf_cache"

drive.mount("/content/drive")
sh(
    f"if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}; else git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}; fi"
)
sh(f"git -C {REPO_DIR} log --oneline -1")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

**Paste back:** the commit line and the GPU line (should say NVIDIA L4).

## 2 · Pinned environment (~3–5 min)

In [ ]:
%%time
sh("pip install -q uv")
sh(f"uv venv -q --clear --python 3.13 {VENV}")
sh(
    f"uv pip install -q --python {PY} -e {REPO_DIR} -r {REPO_DIR}/requirements/colab-gpu.txt -c {REPO_DIR}/requirements/colab-gpu.lock.txt"
)
sh(f"{PY} -m vizwiz.compat versions")

## 3 · Unzip photos (~4 min)
Training and val photos come from `train.zip`; the test set (official val) from `val.zip`.

In [ ]:
%%time
import hashlib

expected = json.load(open(f"{REPO_DIR}/results/splits/files.json"))
for name in ("vqa_test.jsonl", "quality_test.jsonl"):
    digest = hashlib.sha256(open(f"{DATA}/processed/{name}", "rb").read()).hexdigest()
    assert digest == expected[name]["sha256"], f"{name} differs from the committed hash"
    print(f"{name}: matches committed hash")
sh(f"{PY} -m vizwiz.data.download extract --root {DATA} --dest {LOCAL} --splits train val")

## 4 · Train (~3.5–4 h)
The first model download (~9 GB) happens here. Val loss on 500 val examples every 200 steps; the live curve is in W&B (project `vizwiz-assistant`, run `e2-answerability-lora`).

After a disconnect, this same cell resumes from the newest checkpoint. If training already finished, it is skipped.

In [ ]:
%%time
summary_path = f"{CKPT}/{RUN_ID}/train_summary.json"
if os.path.exists(summary_path) and os.path.exists(f"{ADAPTER}/adapter_config.json"):
    print("training already finished: skipped")
else:
    sh(
        f"{PY} -m vizwiz.train.sft train --config {REPO_DIR}/configs/train/{TASK}.yaml --mode lora --records {DATA}/processed --images {LOCAL}/images --out {CKPT} --run-id {RUN_ID} --wandb --wandb-group e2"
    )
s = json.load(open(summary_path))
print(
    "TRAIN DONE:",
    {
        k: s[k]
        for k in (
            "global_step",
            "optimizer_steps_planned",
            "resumed_from",
            "train_loss",
            "final_val_loss",
            "examples_per_s",
            "peak_mem_gb",
            "wall_s",
        )
    },
)

**Paste back:** the `TRAIN DONE: ...` line.

## 5 · Evaluate on the full test set (~10 min)
Same command as E1 plus `--adapter`. Skipped if already done.

In [ ]:
%%time
OUT = f"{RUNS}/{RUN_ID}"
done = (
    os.path.exists(f"{OUT}/metrics.json")
    and TASK in json.load(open(f"{OUT}/metrics.json"))["tasks"]
)
if done:
    print("evaluation already done: skipped")
else:
    sh(
        f"{PY} -m vizwiz.eval.run_eval --records {DATA}/processed --images {LOCAL}/images --split test --tasks {TASK} --adapter {ADAPTER} --run-id {RUN_ID} --out {OUT}"
    )

**Paste back:** the `[answerability] average_precision = ...` line.

## 6 · Compare with E1 (~1 min)

In [ ]:
sh(
    f"{PY} -m vizwiz.eval.compare --records {DATA}/processed --split test --task {TASK} --runs {RUNS}/e1-baseline {OUT} --names e1 e2 --reference e1 --report {OUT}/compare_{TASK}.json > /dev/null"
)
c = json.load(open(f"{OUT}/compare_{TASK}.json"))
print(
    "COMPARE:",
    c["metric"],
    c["n"],
    {k: (r["value"], r["diff_vs_reference"], r["diff_ci95"]) for k, r in c["runs"].items()},
)
e1 = json.load(open(f"{RUNS}/e1-baseline/metrics.json"))["tasks"][TASK]["metrics"]
e2 = json.load(open(f"{OUT}/metrics.json"))["tasks"][TASK]["metrics"]
KEYS = [
    "average_precision",
    "recall_at_precision_0.8",
    "recall_at_precision_0.9",
    "official_f1_at_0.5",
    "invalid_output_pct",
]
for k in KEYS:
    print(f"{k:26s} e1 {e1[k]!s:>7}  e2 {e2[k]!s:>7}")
print("confusion e2:", e2["confusion"])

**Paste back:** everything this cell printed (numbers only).

## 7 · Flush Drive, then disconnect
Upload back (summary numbers only; prediction files and checkpoints stay on Drive):
- `MyDrive/vizwiz/checkpoints/e2-answerability-lora/train_summary.json`
- `MyDrive/vizwiz/checkpoints/e2-answerability-lora/settings.json`
- `MyDrive/vizwiz/runs/e2-answerability-lora/metrics.json`
- `MyDrive/vizwiz/runs/e2-answerability-lora/compare_answerability.json`

Then Runtime → **Disconnect and delete runtime** (an idle L4 keeps using compute units).

Keep `checkpoints/e2-answerability-lora/adapter/` on Drive: it is the trained adapter. The two `checkpoint-*` folders next to it (~0.8 GB) can be deleted once the numbers look right.

In [ ]:
drive.flush_and_unmount()
print("Drive flushed. Safe to disconnect.")